# Knowledge-Based Decision Support System (DSS)
## Context-Aware Music Recommendation & Routine Optimization

**Student:** Stefano Colella (865075)  
**Course:** Fundamentals of AI (9 CFU) — Master's Degree in Computer Science, University of Bari Aldo Moro

---

### Project Abstract
Conventional music recommenders ignore **how the listener actually feels** in real time. This **Knowledge-Based Decision Support System (DSS)** combines:
1. **Spotify Audio Features** (`data/spotify_tracks.json`: BPM, valence, energy, instrumentalness, semantic tags).
2. **Smartwatch Telemetry** (`data/smartwatch_biometrics.json`: heart rate, stress index, current activity, routine goal).

Following a classic **Expert System architecture** (separated **Facts**, **Declarative Rules + Taxonomy + Askables**, **Working Memory**, and **Meta-Interpreter** with **HOW/WHY** explanations and uncertainty degrees in $[0, 1]$), Python acts purely as an interface and verification driver over the **Prolog inference engine**.

## 1. Dataset Inspection
Inspection of the structured JSON catalogs in `data/`.

In [1]:
import json

with open('data/spotify_tracks.json', 'r', encoding='utf-8') as f:
    tracks = json.load(f)
with open('data/smartwatch_biometrics.json', 'r', encoding='utf-8') as f:
    users = json.load(f)

print(f"[+] Loaded {len(tracks)} Spotify tracks and {len(users)} smartwatch user profiles.")
print("\n--- Smartwatch User Profiles ---")
for u in users:
    print(f" * {u['name']} ({u['user_id']}) -> Activity: {u['current_activity']} | HR: {u['heart_rate_bpm']} BPM | Stress: {u['stress_level']}/100 | Goal: {u['target_health_goal']}")

[+] Loaded 30 Spotify tracks and 6 smartwatch user profiles.

--- Smartwatch User Profiles ---
 * Sarah Miller (user_sarah) -> Activity: high_stress_work | HR: 114 BPM | Stress: 84/100 | Goal: stress_downregulation
 * Alex Chen (user_alex) -> Activity: cardio_running | HR: 146 BPM | Stress: 45/100 | Goal: cardio_peak_performance
 * Marco Rossi (user_marco) -> Activity: deep_focus_study | HR: 74 BPM | Stress: 32/100 | Goal: deep_focus_flow
 * Elena Rostova (user_elena) -> Activity: post_workout_recovery | HR: 122 BPM | Stress: 48/100 | Goal: post_workout_recovery
 * David Becker (user_david) -> Activity: sleep_preparation | HR: 68 BPM | Stress: 22/100 | Goal: sleep_induction
 * Clara Dupont (user_clara) -> Activity: active_commute | HR: 96 BPM | Stress: 38/100 | Goal: active_commute_energizer


## 2. Knowledge Base Fact Compilation (`generate_kb.py` -> `knowledge_base.pl`)
The ETL script translates the JSON catalogs into normalized Prolog facts (`track/8`, `track_tag/2`, `user/7`), keeping factual data decoupled from domain rules (`kb_rules.pl`) and inference control (`inference_engine.pl`).

In [2]:
import generate_kb
generate_kb.main()

with open('knowledge_base.pl', 'r', encoding='utf-8') as f:
    lines = [next(f) for _ in range(16)]
print(''.join(lines))

[*] Loading raw JSON datasets...
[*] Emitting Prolog facts to knowledge_base.pl...
[+] Knowledge base compiled successfully: 30 tracks, 94 tags, 6 user profiles written to knowledge_base.pl.
% =========================================================================
% KNOWLEDGE BASE FACTS: knowledge_base.pl
% Automatically generated by generate_kb.py from JSON datasets
% =========================================================================

% --- Track Catalog: track(Id, Title, Artist, Genre, BPM, Valence, Energy, Instrumentalness) ---
:- dynamic track/8.

track(weightless_ambient, 'Weightless Ambient Drift', 'NeuroHarmonics', ambient, 60.0, 0.35, 0.12, 0.96).
track(clair_de_lune, 'Clair de Lune (Debussy)', 'Sinfonia Chambers', classical, 64.0, 0.42, 0.18, 0.94).
track(delta_sleep_stream, 'Delta Wave Binaural Serenade', 'SomnoLab', ambient, 55.0, 0.25, 0.08, 0.99).
track(gymnopedie_no1, 'Gymnopedie No. 1 (Satie)', 'Aura Piano Quartet', classical, 66.0, 0.38, 0.15, 0.95).
track(peac

## 3. Live Prolog Inference & HOW Proof Trees
Below, Python directly invokes the Prolog inference engine (`main.pl`) to evaluate user profiles and print the actual **HOW** proof tree constructed by the meta-interpreter (`inference_engine.pl`).

In [3]:
from test_suite import find_prolog_engine, run_prolog_goal, fetch_prolog_results

engine_name, engine_bin = find_prolog_engine()
print(f"[*] Executing queries on Prolog engine: {engine_name.upper()} ({engine_bin})\n")

best, playlist, _, _ = fetch_prolog_results()

for uid, label in [
    ('user_sarah', 'CASE 1: High Work Stress & Safety Protection'),
    ('user_alex',  'CASE 2: Intense Running Workout'),
    ('user_marco', 'CASE 3: Lyric-Free Study Focus'),
    ('user_david', 'CASE 4: Pre-Sleep Routine')
]:
    rec = best[uid]
    print('=' * 72)
    print(f' {label} ({uid})')
    print('=' * 72)
    print(f" -> RECOMMENDED: '{rec['title']}' by {rec['artist']} [{rec['genre']}]")
    print(f" -> Utility: {rec['score']:.1f}/100 | Rule Confidence: {rec['confidence']:.2f}")
    print(f" -> Audio: {rec['bpm']:.0f} BPM | Energy: {rec['energy']:.2f} | Valence: {rec['valence']:.2f} | Instrumental: {rec['instrumentalness']:.2f}\n")

print('--- Real Prolog HOW Proof Tree for Sarah Miller ---')
print(run_prolog_goal('recommend_with_how(user_sarah)'))

[*] Executing queries on Prolog engine: YAP (C:\Program Files (x86)\Yap\bin\yap.exe)

 CASE 1: High Work Stress & Safety Protection (user_sarah)
 -> RECOMMENDED: 'Clair de Lune (Debussy)' by Sinfonia Chambers [classical]
 -> Utility: 90.3/100 | Rule Confidence: 0.94
 -> Audio: 64 BPM | Energy: 0.18 | Valence: 0.42 | Instrumental: 0.94

 CASE 2: Intense Running Workout (user_alex)
 -> RECOMMENDED: 'Subterranean Stamina Loop' by Klangwerk [techno]
 -> Utility: 88.5/100 | Rule Confidence: 0.96
 -> Audio: 140 BPM | Energy: 0.92 | Valence: 0.62 | Instrumental: 0.90

 CASE 3: Lyric-Free Study Focus (user_marco)
 -> RECOMMENDED: 'Analog Cassette Memory' by BeatFoundry [lofi]
 -> Utility: 91.1/100 | Rule Confidence: 0.90
 -> Audio: 80 BPM | Energy: 0.38 | Valence: 0.58 | Instrumental: 0.86

 CASE 4: Pre-Sleep Routine (user_david)
 -> RECOMMENDED: 'Delta Wave Binaural Serenade' by SomnoLab [ambient]
 -> Utility: 89.9/100 | Rule Confidence: 0.93
 -> Audio: 55 BPM | Energy: 0.08 | Valence: 0.25 |

## 4. Gradual Calm Playlist (3-Step Smooth Transition)
When a user is under high stress (e.g., 114 BPM), playing a very slow track immediately can feel jarring. `gradual_calm_playlist/2` searches for an optimal 3-step downward tempo curve: **Entry -> Bridge -> Target**.

In [4]:
print(run_prolog_goal('recommend_session(user_sarah)'))


+======================================================================+
| 3-STEP GRADUAL CALM PLAYLIST (ENTRY -> BRIDGE -> TARGET)             |
+======================================================================+
| User: Sarah Miller | Heart Rate: 114 BPM | Stress: 84/100
+----------------------------------------------------------------------+
| 1. [ENTRY]  Ambient Horizon (105 BPM, Energy: 0.55)
| 2. [BRIDGE] Neural Flowstate Synapses (92 BPM, Energy: 0.50)
| 3. [TARGET] Delta Wave Binaural Serenade (55 BPM, Energy: 0.08)
+----------------------------------------------------------------------+
| Trajectory: 105 BPM -> 92 BPM -> 55 BPM (Smooth Calming Curve)
+======================================================================+




## 5. Automated 10-Point Prolog Verification Suite (`test_suite.py`)
Executes the full verification suite directly against the Prolog inference engine.

In [5]:
import test_suite
test_suite.main()


 PROLOG-DRIVEN VERIFICATION SUITE (SEPARATED KB + INFERENCE ENGINE)
 Active Prolog Interpreter: YAP (C:\Program Files (x86)\Yap\bin\yap.exe)
 [+] 1. Stress Relief Pillar (Sarah) -> Slow calm instrumental track: PASSED
 [+] 2. Intense Workout Pillar (Alex) -> Fast beat & high energy: PASSED
 [+] 3. Study Lyric-Free Pillar (Marco) -> Steady instrumental focus track: PASSED
 [+] 4. Pre-Sleep Pillar (David) -> Very slow & very low energy: PASSED
 [+] 5. Post-Workout Recovery (Elena) -> Balanced moderate pacing: PASSED
 [+] 6. Active Commute Vitality (Clara) -> Positive mood & moderate beat: PASSED
 [+] 7. Gradual Calm Playlist -> Monotonic BPM descent (Entry -> Bridge -> Target): PASSED
 [+] 8. Safety Barrier (Negative Test) -> Block fast/rock track for stressed Sarah: PASSED
 [+] 9. Study Barrier (Negative Test) -> Block vocal pop track for Marco: PASSED
 [+] 10. Multi-Criteria Utility & Genre Preference -> Preferred genre scores higher: PASSED
-------------------------------------------